# 03 — Query & Generate

**Audience:** Engineers / SEs following the technical walkthrough.

**What this notebook shows:**

1. **Hybrid retrieval, three ways.** We run the same question through
   `retrieve(..., mode="sentences")`, `mode="chunks"`, and `mode="dual"`,
   and render each as a table so you can *see* how the candidate pool
   and the context window differ.
2. **The dual-layout payoff.** `mode="dual"` unions Layout Y's sentence
   candidates with Layout X's chunk context (and pulls in the parent
   chunk of every sentence hit so every citation target has full
   surrounding text). This is the single most important shape in the
   codebase — everything downstream depends on it.
3. **Generation — both strategies.**
   - Strategy A (`inline_prompted`): the RAG model emits `[s:<sid>]`
     markers inline as it writes.
   - Strategy B (`post_gen_alignment`): the model writes a clean
     answer, then `cite_align` attaches citations after the fact.
4. **Side-by-side comparison.** Two questions × two strategies, one
   table, so the trade-offs are legible at a glance.

**Prerequisites:**

- `02_chunk_and_index.ipynb` completed — **both** indexes
  (`tax-chunks` and `tax-sentences`) must be live in Azure AI Search.
  02 builds them.
- `.env` populated; `DefaultAzureCredential` can reach Search,
  embeddings, and the Foundry chat deployment.
- Or: set `RESUME_FROM_CACHE = True` below to load a retrieval
  snapshot from `data/notebook_cache/retrieval/` and skip live calls.
  03 is primarily a live-call walkthrough, so the default is `False`.

The dual-layout rationale lives in
[`docs/index_projections_eval.md`](../docs/index_projections_eval.md).
Strategy-level design notes are in `src/sentcite/generate.py` and
`src/sentcite/cite_align.py`.

In [ ]:
# --- Top-level knobs --------------------------------------------------------
# Flip to True to load retrieval snapshots from data/notebook_cache/retrieval/
# instead of hitting Azure Search. 03 is a live-call walkthrough by default
# because seeing the three retrieval shapes light up is the whole point.
RESUME_FROM_CACHE = False

# Cost/scale knob. Each question = 1 embeddings call + up to 2 Search calls
# + (for generation) 1 chat completion per strategy. Keep this tiny while you
# iterate; scale up once the shapes look right.
LIMIT_QUESTIONS = 2

# Retrieval depth. top_k=5 keeps the chunk context small enough to eyeball
# and matches the default k_chunks in sentcite.retrieval.retrieve.
TOP_K = 5

# Sample questions — drawn from the Phase 1a synthetic GT over IRS pubs.
# Mix of easy / medium so the retrieval tables show some texture.
SAMPLE_QUESTIONS = [
    # Pub 587, Area Adjustment Worksheet.
    "What single-letter line item in Part II of the Area Adjustment Worksheet must be completed if the business use area of the home changed during the year?",
    # Pub 463, actual cost method.
    "What is required of a taxpayer who chooses the actual cost method for deducting expenses under Publication 463?",
    # Pub 17, life-insurance-in-retirement-plan edge case.
    "Under what circumstance might the cost of life insurance coverage in a retirement plan be taxable to an individual?",
]

In [ ]:
import json, sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / 'src'))

load_dotenv(REPO_ROOT / '.env')

from sentcite.config import AzureConfig
from sentcite.retrieval import retrieve
from sentcite.generate import generate, build_context_block
from sentcite.cite_align import cite_answer
from sentcite.llm import get_binding

cfg = AzureConfig.from_env()
rag = get_binding('rag', cfg)

CACHE_DIR = REPO_ROOT / 'data/notebook_cache/retrieval'

{
    'rag_model': rag.model_identity,
    'rag_deployment': rag.deployment,
    'chunks_index': cfg.search_index_chunks,
    'sentences_index': cfg.search_index_sentences,
    'resume_from_cache': RESUME_FROM_CACHE,
    'cache_dir_exists': CACHE_DIR.exists(),
}

In [ ]:
# Best-effort cache shim. RESUME_FROM_CACHE=True would load a previously
# serialized RetrievalResult per (question, mode) from data/notebook_cache/
# retrieval/<slug>__<mode>.json and skip the live Search call.
#
# TODO: the serialization format isn't finalized yet — when it is, plug the
# deserializer in here. For now we just fall through to a live retrieve().
def retrieve_cached(question: str, *, mode: str, top_k: int):
    if RESUME_FROM_CACHE and CACHE_DIR.exists():
        # TODO: hook up real cache load once shape is frozen.
        pass
    return retrieve(question, cfg=cfg, mode=mode, k_chunks=top_k, k_sentences=top_k * 4)

## Part 1 — Retrieval tour

This is the meat of the notebook. Before we ever generate an answer
we need to understand what the generator is going to be staring at.

**Two indexes, two shapes:**

- **Layout X — `tax-chunks`.** One document per ~400-token chunk.
  Chunks carry a nested `sentences` array so the sentence provenance
  travels with the chunk. This is the generator's **context**
  surface — hybrid + semantic search returns the chunks whose full
  text we'll paste into the prompt.
- **Layout Y — `tax-sentences`.** One document per sentence,
  materialized via index projections from Layout X. This is the
  **citation-candidate** surface — hybrid + semantic search returns
  ranked sentence IDs that are valid citation targets.

**Three modes of `retrieve(...)`:**

| mode          | chunk context                    | sentence candidates          | use |
|---------------|----------------------------------|------------------------------|-----|
| `"chunks"`    | Layout X top-k                   | nested sentences of those X  | Layout-X-only ablation |
| `"sentences"` | — (no chunk context)              | Layout Y top-k               | Layout-Y-only ceiling/floor |
| `"dual"`      | Layout X top-k **∪ parents of Y** | Layout Y top-k               | **production default** |

We'll now run one sample question through all three and look at the
tables side by side.

In [ ]:
# Pick one question for the retrieval tour. We'll reuse this across all
# three modes so the comparison is apples-to-apples.
Q = SAMPLE_QUESTIONS[0]
Q

### 1a. `mode="sentences"` — Layout Y only

Every hit is a standalone sentence. No surrounding chunk context. Good for measuring the **ceiling** of sentence-level recall, but the generator would be writing from sentence fragments.

In [ ]:
res_sent = retrieve_cached(Q, mode='sentences', top_k=TOP_K)

sent_df = pd.DataFrame([
    {
        'sentence_id': s.sentence_id,
        'parent_chunk_id': s.chunk_id,
        'page': s.page,
        'reranker': round(s.reranker, 3) if s.reranker is not None else None,
        'score': round(s.score, 3) if s.score is not None else None,
        'text[:200]': (s.text or '')[:200],
    }
    for s in res_sent.sentence_candidates[:TOP_K]
])

print(f'sentences mode: {len(res_sent.sentence_candidates)} candidates, '
      f'{len(res_sent.chunks)} chunks in context, '
      f'latency={res_sent.latency_ms:.0f} ms')
sent_df

### 1b. `mode="chunks"` — Layout X only

Top-k chunks by hybrid + semantic search. The nested sentences of these chunks are the *only* citable sentences. If the relevant sentence lives in a chunk that didn't rank top-k, you can't cite it.

In [ ]:
res_chunk = retrieve_cached(Q, mode='chunks', top_k=TOP_K)

chunk_df = pd.DataFrame([
    {
        'chunk_id': c.chunk_id,
        'page': c.page,
        'section': ' > '.join(c.section_path)[-80:],
        'n_sentences': len(c.sentences),
        'source': c.source,
        'reranker': round(c.reranker, 3) if c.reranker is not None else None,
        'score': round(c.score, 3) if c.score is not None else None,
        'text[:200]': (c.text or '')[:200],
    }
    for c in res_chunk.chunks
])

print(f'chunks mode: {len(res_chunk.chunks)} chunks, '
      f'{len(res_chunk.sentence_candidates)} candidate sentences '
      f'(nested), latency={res_chunk.latency_ms:.0f} ms')
chunk_df

### 1c. `mode="dual"` — the union (production default)

`dual` runs **both** searches and then unions the parent chunks of the
sentence candidates into the chunk context. Result:

- **Every sentence candidate has its full parent chunk available** as
  context for the generator. No orphaned citations.
- **Chunk-level ranking still surfaces chunks** whose best sentence
  may not have topped Layout Y on its own.

The `source` column on each chunk tells you how it got into the
context:

- `chunk_search` — only Layout X found it.
- `sentence_parent` — only a Layout Y sentence candidate pulled it in.
- `both` — Layout X ranked it **and** a Y candidate lives inside it
  (strongest signal).

This is the shape `generate(...)` consumes.

In [ ]:
res_dual = retrieve_cached(Q, mode='dual', top_k=TOP_K)

dual_chunk_df = pd.DataFrame([
    {
        'chunk_id': c.chunk_id,
        'source': c.source,
        'page': c.page,
        'section': ' > '.join(c.section_path)[-80:],
        'n_sentences': len(c.sentences),
        'reranker': round(c.reranker, 3) if c.reranker is not None else None,
        'score': round(c.score, 3) if c.score is not None else None,
    }
    for c in res_dual.chunks
])

dual_sent_df = pd.DataFrame([
    {
        'sentence_id': s.sentence_id,
        'parent_chunk_id': s.chunk_id,
        'parent_in_context': s.chunk_id in {c.chunk_id for c in res_dual.chunks},
        'reranker': round(s.reranker, 3) if s.reranker is not None else None,
        'text[:160]': (s.text or '')[:160],
    }
    for s in res_dual.sentence_candidates[:TOP_K]
])

print(
    f'dual mode: {res_dual.chunk_search_hits} chunk hits + '
    f'{res_dual.parent_chunks_added} parent chunks added = '
    f'{len(res_dual.chunks)} chunks in context | '
    f'{len(res_dual.sentence_candidates)} sentence candidates | '
    f'latency={res_dual.latency_ms:.0f} ms'
)
dual_chunk_df

In [ ]:
# Sentence candidates in dual mode — note parent_in_context is True for
# every row. That's the invariant the union is designed to enforce.
dual_sent_df

### 1d. Why `dual` is the default

Look at the three tables together:

- **`sentences` alone** gives you the most precise citation targets
  but no context for the generator. A model asked to answer from
  five disconnected sentences will either refuse or hallucinate the
  connective tissue.
- **`chunks` alone** gives the generator plenty to read but ties
  citations to chunk-level retrieval quality. If the most-relevant
  *sentence* lives in a chunk that ranked 6th, it's unreachable.
- **`dual`** pays for one extra Search call (the parent-chunk
  hydration) and gets both: Layout Y's ranked sentence candidates
  for citation, with every one of them backed by its full parent
  chunk in the context window.

Every evaluation number downstream — faithfulness, alignment,
answer-correctness — assumes this shape. The rest of the notebook
generates against `res_dual`.

## Part 2 — Generation, Strategy A: `inline_prompted`

The RAG model sees every source sentence prefixed with its
`[s:<sentence_id>]` tag (see `build_context_block(..., include_sentence_ids=True)`)
and is instructed to append the matching tag after every claim it
writes.

**Pros:** the model is context-aware when it cites — it can cite
multiple sources on one sentence, or the same source twice, naturally.

**Cons:** the model can (a) skip citing a supported claim,
(b) cite a plausible-looking id that isn't actually in the
whitelist, or (c) wander outside the sources entirely. The
whitelist check in `parse_inline_citations` catches (b); (a) and
(c) are why we also measure faithfulness and alignment in later
notebooks.

In [ ]:
# Peek at what the model actually sees for Strategy A — the context block
# with explicit [s:<sid>] prefixes on every source sentence.
ctx_text, ctx_sids = build_context_block(res_dual, include_sentence_ids=True)
print(f'context: {len(ctx_text)} chars, {len(ctx_sids)} taggable sentences')
print('---')
print(ctx_text[:1200] + ('\n...' if len(ctx_text) > 1200 else ''))

In [ ]:
gen_A = generate(Q, res_dual, strategy='inline_prompted', cfg=cfg)

print(f'model={gen_A.model}  finish={gen_A.finish_reason}  '
      f'latency={gen_A.latency_ms:.0f} ms  '
      f'prompt_toks={gen_A.prompt_tokens}  completion_toks={gen_A.completion_tokens}')
print('---')
print(gen_A.answer_text)

In [ ]:
# Dispatch Strategy A through cite_answer → this runs parse_inline_citations
# under the hood: strip the [s:...] tags to get clean sentences, attribute
# each tag to the sentence it fell inside, drop anything not in the
# whitelist.
cited_A = cite_answer(gen_A, res_dual, cfg=cfg)

pd.DataFrame([
    {
        'i': s.index,
        'sentence': s.text,
        'n_citations': len(s.citations),
        'sentence_ids': [c.sentence_id for c in s.citations],
        'source': s.citations[0].source if s.citations else None,
    }
    for s in cited_A.sentences
])

## Part 3 — Generation, Strategy B: `post_gen_alignment`

The RAG model sees the same chunks as prose — **no sentence-id
prefixes** (see `build_context_block(..., include_sentence_ids=False)`)
— and is explicitly told *not* to emit citation markers. We get a
clean, readable answer. Then `cite_align.align_post_generation`
embeds each answer sentence, cosine-scores it against the Layout Y
candidate pool, and attaches citations above a similarity
threshold (`τ = 0.75` by default).

**Pros:** clean answer text, deterministic alignment, same code path
regardless of which RAG model produced the answer.

**Cons:** all the citation intelligence is now in `cite_align`.
Threshold tuning, top-k per answer sentence, and the embedding
model itself are all knobs that matter. `04_citation_alignment.ipynb`
opens that black box.

In [ ]:
gen_B = generate(Q, res_dual, strategy='post_gen_alignment', cfg=cfg)

print(f'model={gen_B.model}  finish={gen_B.finish_reason}  '
      f'latency={gen_B.latency_ms:.0f} ms  '
      f'prompt_toks={gen_B.prompt_tokens}  completion_toks={gen_B.completion_tokens}')
print('---')
print(gen_B.answer_text)

In [ ]:
# Light-touch cite_align: align_post_generation embeds each answer
# sentence, cosine-scores it against res_dual.sentence_candidates, and
# keeps matches above tau. 04 goes deep — here we just show the shape.
cited_B = cite_answer(gen_B, res_dual, cfg=cfg)

pd.DataFrame([
    {
        'i': s.index,
        'sentence': s.text,
        'n_citations': len(s.citations),
        'top_match': s.citations[0].sentence_id if s.citations else None,
        'top_confidence': round(s.citations[0].confidence, 3) if s.citations else None,
        'source': s.citations[0].source if s.citations else None,
    }
    for s in cited_B.sentences
])

## Part 4 — Side-by-side: 2 questions × 2 strategies

Same retrieval (`mode="dual"`) for both strategies. What differs is
how citations get attached.

In [ ]:
def _render_with_citations(cited) -> str:
    '''Render a CitedAnswer as 'sentence [sid1, sid2] sentence [sid3] ...'.'''
    parts = []
    for s in cited.sentences:
        if s.citations:
            ids = ', '.join(c.sentence_id for c in s.citations)
            parts.append(f'{s.text} [{ids}]')
        else:
            parts.append(s.text)
    return ' '.join(parts)


rows = []
for q in SAMPLE_QUESTIONS[:LIMIT_QUESTIONS]:
    r = retrieve_cached(q, mode='dual', top_k=TOP_K)
    for strat in ('inline_prompted', 'post_gen_alignment'):
        g = generate(q, r, strategy=strat, cfg=cfg)
        ca = cite_answer(g, r, cfg=cfg)
        rows.append({
            'question': q[:70] + ('…' if len(q) > 70 else ''),
            'strategy': strat,
            'answer_with_citations': _render_with_citations(ca),
            'num_citations': sum(len(s.citations) for s in ca.sentences),
            'num_answer_sentences': len(ca.sentences),
        })

compare_df = pd.DataFrame(rows)
compare_df

In [ ]:
# Readable dump — DataFrame truncates long strings.
for r in rows:
    print(f"[{r['strategy']}] {r['question']}")
    print(f"  {r['answer_with_citations']}")
    print(f"  ({r['num_citations']} citations across {r['num_answer_sentences']} sentences)")
    print()

## What's next

- **`04_citation_alignment.ipynb`** — opens the `cite_align` black
  box. We'll tune `τ`, inspect the cosine matrix answer-sentence ×
  candidate, see where Strategy B disagrees with Strategy A, and
  look at the failure modes (under-cited claims, over-cited filler,
  whitelist misses).
- **`05_eval.ipynb`** — end-to-end numbers over the synthetic GT:
  faithfulness, citation precision/recall, answer correctness, plus
  the per-mode ablation (`dual` vs `chunks` vs `sentences`) using
  this exact retrieval surface.